# Información mutua y entropía condicional

Este cuaderno acompaña el artículo sobre información mutua. Trabajaremos con distribuciones conjuntas pequeñas para ver cómo cambian `H(X)`, `H(X | Y)` e `I(X; Y)`.

La idea central es que la información mutua mide cuánta incertidumbre sobre una variable desaparece al observar otra.

In [ ]:
import math


def entropia(distribucion):
    return -sum(p * math.log2(p) for p in distribucion.values() if p > 0)


def marginal_x(conjunta):
    resultado = {}
    for (x, _), p in conjunta.items():
        resultado[x] = resultado.get(x, 0) + p
    return resultado


def marginal_y(conjunta):
    resultado = {}
    for (_, y), p in conjunta.items():
        resultado[y] = resultado.get(y, 0) + p
    return resultado


def entropia_condicional_x_dado_y(conjunta):
    py = marginal_y(conjunta)
    total = 0
    for y, prob_y in py.items():
        distribucion = {x: p / prob_y for (x, y2), p in conjunta.items() if y2 == y}
        total += prob_y * entropia(distribucion)
    return total


def informacion_mutua(conjunta):
    return entropia(marginal_x(conjunta)) - entropia_condicional_x_dado_y(conjunta)

## Tres casos pequeños

Comparamos independencia, copia perfecta y copia con ruido.

In [ ]:
independientes = {
    (0, 0): 0.25,
    (0, 1): 0.25,
    (1, 0): 0.25,
    (1, 1): 0.25,
}

copia_perfecta = {
    (0, 0): 0.5,
    (0, 1): 0.0,
    (1, 0): 0.0,
    (1, 1): 0.5,
}

copia_con_ruido = {
    (0, 0): 0.45,
    (0, 1): 0.05,
    (1, 0): 0.05,
    (1, 1): 0.45,
}

casos = {
    "independientes": independientes,
    "copia perfecta": copia_perfecta,
    "copia con ruido": copia_con_ruido,
}

for nombre, conjunta in casos.items():
    hx = entropia(marginal_x(conjunta))
    hxy = entropia_condicional_x_dado_y(conjunta)
    ixy = informacion_mutua(conjunta)
    print(f"{nombre:16s} H(X) = {hx:.4f}  H(X|Y) = {hxy:.4f}  I(X;Y) = {ixy:.4f}")

## Canal binario con ruido

Si `X` es uniforme y `Y` es una copia de `X` con probabilidad de error `p`, la información mutua baja a medida que sube el ruido.

In [ ]:
def conjunta_canal_binario(p):
    return {
        (0, 0): 0.5 * (1 - p),
        (0, 1): 0.5 * p,
        (1, 0): 0.5 * p,
        (1, 1): 0.5 * (1 - p),
    }


for p in [0, 0.01, 0.05, 0.1, 0.2, 0.5]:
    conjunta = conjunta_canal_binario(p)
    print(f"p = {p:>4.2f}  I(X;Y) = {informacion_mutua(conjunta):.4f} bits")

In [ ]:
probabilidades = [i / 100 for i in range(0, 51)]
mutuas = [informacion_mutua(conjunta_canal_binario(p)) for p in probabilidades]

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib no está instalado; se omite la gráfica.")
else:
    plt.figure(figsize=(7, 4))
    plt.plot(probabilidades, mutuas)
    plt.xlabel("Probabilidad de error p")
    plt.ylabel("I(X;Y) en bits")
    plt.title("Información mutua en un canal binario simétrico")
    plt.grid(True, alpha=0.3)
    plt.show()

## Para experimentar

1. Modifica `copia_con_ruido` para que el ruido sea mayor.
2. Cambia la distribución de `X` para que no sea uniforme.
3. Comprueba que `I(X;Y)` nunca supera a `H(X)`.